In [1]:
import numpy as np

In [2]:
def hamiltonian_matrix(delta_x,potential_values):
  potential_values = np.asarray(potential_values,dtype=float)
  if delta_x <=0:
    raise ValueError("detla_x must be greater than 0")
  if potential_values.ndim != 1:
    raise ValueError("potential_values must be a one-dimensional array.")
  if potential_values.size == 0:
    raise ValueError("potential_values cannot be empty.")

  interior_points = len(potential_values)

  main_diag = 1/(delta_x**2) + potential_values
  off_diag = np.full(interior_points-1,-1/(2*(delta_x**2)))

  hamiltonian = (np.diag(main_diag) + np.diag(off_diag,k=1) + np.diag(off_diag,k=-1))

  return hamiltonian

In [3]:
def solve_and_plot(xi,xf,interior_points,state_index,potential_values):
  delta_x = (xf - xi)/(interior_points + 1)
  interior_grid = (xi + delta_x*np.arange(1,interior_points+1))

  potential_values = np.asarray(potential_values,dtype=float)

  if state_index < 0 or state_index >= interior_points:
    raise ValueError("state_index must lie between 0 and interior_points - 1.")

  H = hamiltonian_matrix(delta_x,potential_values)

  eigenvalues,eigenvectors = np.linalg.eigh(H)

  phi_interior   = eigenvectors[:,state_index]
  energy_value   = eigenvalues[state_index]
  phi            = np.concatenate(([0.0],phi_interior,[0.0]))
  normalization  = np.sum(np.abs(phi)**2)*delta_x
  phi_normalized = phi/np.sqrt(normalization)
  full_grid      = np.concatenate(([xi],interior_grid,[xf]))
  quantum_number = state_index + 1
  check = np.sum(np.abs(phi_normalized)**2) * delta_x
  print("Normalization =", check)

  plt.figure(figsize=(8, 5))
  plt.plot(full_grid,phi_normalized,label=f"State {quantum_number}")
  plt.axhline(0,linewidth=0.8,linestyle="--")
  plt.xlabel("Dimensionless position, x")
  plt.ylabel(r"Eigenvector $\phi(x)$")
  plt.title(f"Eigenvalue n = {quantum_number}, "f"Energy = {energy_value:.6f}")
  plt.grid(alpha=0.3)
  plt.legend()
  plt.show()

  return full_grid,phi,energy_value